In [1]:
from typing import TypedDict ,List ,Union
from langchain_core.messages import HumanMessage ,AIMessage
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph ,START ,END
from dotenv import load_dotenv
from typing import Annotated , Sequence ,TypedDict
from langchain_core.messages import BaseMessage ,ToolMessage ,SystemMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph  ,END ,START
from langgraph.prebuilt import ToolNode
from langchain.chat_models import init_chat_model

In [29]:
import os
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser

In [30]:
llm = init_chat_model(
    model="llama3.1:latest ",#phi3.5",  # The name of the model in Ollama #'deepseek-r1:1.5b',#
    model_provider="ollama",  # Explicitly state the provider
    temperature=0.7,  # Optional: control creativity
    # base_url="http://localhost:11434" # Optional: If your Ollama is on a different host/port
)

In [31]:
PDF_PATH = 'The styrene production.pdf'#'Production_of_Cumene_from_Benzene_Propylene.pdf'#"eg_asp.pdf"#"IPA1.pdf"
LLM_MODEL = "llama3.1:latest"
EMBEDDING_MODEL = "mxbai-embed-large"

In [38]:


def setup_rag_chain(pdf_path: str):
    """
    Sets up a RAG chain by loading a PDF, creating/updating a ChromaDB vector store,
    and initializing the LangChain components.

    Args:
        pdf_path (str): The file path to the PDF document.

    Returns:
        The configured RAG chain.
    """
    # Create a directory name based on the PDF file name
    db_name = os.path.splitext(os.path.basename(pdf_path))[0]
    persist_directory = f"./{db_name}_rag_db"

    # Check if the vector store already exists
    if os.path.exists(persist_directory) and os.listdir(persist_directory):
        print(f"A RAG vector store for '{db_name}' already exists at '{persist_directory}'.")
        choice = input("Do you want to (a)dd to it, (r)emove and replace it, or (c)ontinue with old RAG? (a/r/c): ").lower()
        
        if choice == 'r':
            print(f"Removing old vector store and creating a new one...")
            # Remove existing directory and its contents
            shutil.rmtree(persist_directory)
            print("Old vector store removed.")
            
            # Load and process the new PDF
            loader = PyPDFLoader(pdf_path)
            documents = loader.load()
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
            docs = text_splitter.split_documents(documents)

            if not docs:
                print("Error: The PDF file is empty or could not be processed.")
                return None
            
            embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
            vector_store = Chroma.from_documents(
                documents=docs,
                embedding=embeddings,
                persist_directory=persist_directory
            )
        elif choice == 'a':
            print("Adding documents to the existing vector store...")
            embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
            # Load the existing vector store
            vector_store = Chroma(
                persist_directory=persist_directory,
                embedding_function=embeddings
            )
            # Load and process the new PDF to add to the existing store
            loader = PyPDFLoader(pdf_path)
            documents = loader.load()
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
            docs = text_splitter.split_documents(documents)

            if not docs:
                print("Error: The new PDF file is empty or could not be processed.")
                return None
            
            vector_store.add_documents(documents=docs)
            print("Documents added successfully.")
        elif choice == 'c':
            print("Continuing with the existing vector store...")
            embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
            # Load the existing vector store without making changes
            vector_store = Chroma(
                persist_directory=persist_directory,
                embedding_function=embeddings
            )
        else:
            print("Invalid choice. Operation cancelled.")
            return None
    else:
        print(f"Creating a new ChromaDB vector store for '{db_name}' at '{persist_directory}'...")
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
        docs = text_splitter.split_documents(documents)

        if not docs:
            print("Error: The PDF file is empty or could not be processed.")
            return None

        embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
        vector_store = Chroma.from_documents(
            documents=docs,
            embedding=embeddings,
            persist_directory=persist_directory
        )
        print("New vector store created successfully.")

    retriever = vector_store.as_retriever(search_kwargs={"k": 5})
    llm = Ollama(model=LLM_MODEL)

    prompt = ChatPromptTemplate.from_template("""
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know. 
    
    Question: {question}
    Context: {context}
    Answer:
    """)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

# Example usage:
# if __name__ == "__main__":
#     # Replace 'your_document.pdf' with the path to your PDF
#     rag_chain = setup_rag_chain("Abstract1_StyreneProduction.pdf")
#     if rag_chain:
#         question = "What is the desired product flow rate?"
#         answer = rag_chain.invoke(question)
#         print(f"Question: {question}")
#         print(f"Answer: {answer}")


In [39]:
# def setup_rag_chain(pdf_path: str):
#     # Load the PDF document
#     loader = PyPDFLoader(pdf_path)
#     documents = loader.load()

#     text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
#     docs = text_splitter.split_documents(documents)
    
#     if not docs:
#         print("Error: The PDF file is empty or could not be processed.")
#         return None

#     embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

#     # Store the chunks and embeddings in ChromaDB
#     print("Step 4: Creating a ChromaDB vector store...")
#     vector_store = Chroma.from_documents(documents=docs, embedding=embeddings)
    
#     # Create a retriever
#     retriever = vector_store.as_retriever(search_kwargs={"k": 5})

#     llm = Ollama(model=LLM_MODEL)

#     # Create the prompt template
#     prompt = ChatPromptTemplate.from_template("""
#     You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.
#     If you don't know the answer, just say that you don't know. 
    
#     Question: {question}
#     Context: {context}
#     Answer:
#     """)


#     def format_docs(docs):
#         return "\n\n".join(doc.page_content for doc in docs)

#     rag_chain = (
#         {"context": retriever | format_docs, "question": RunnablePassthrough()}
#         | prompt
#         | llm
#         | StrOutputParser()
#     )

#     return rag_chain

In [40]:
def chat_with_pdf():
    """
    Main function to run the PDF chatbot.
    """
    rag_chain = setup_rag_chain(PDF_PATH)
    
    if rag_chain is None:
        return

    print("\n-------------------------------------------")
    print(f"Chatbot is ready. You can now ask questions about '{os.path.basename(PDF_PATH)}'.")
    print("Type 'exit' to quit.")
    print("-------------------------------------------")
    
    while True:
        user_question = input("\nYour question: ")
        if user_question.lower() == 'exit':
            print("Goodbye!")
            break
        
        print("Thinking...")
        try:
            response = rag_chain.invoke(user_question)
            print(f"\nChatbot: {response}")
        except Exception as e:
            print(f"An error occurred: {e}")
            print("Please make sure the Ollama server is running and the models are pulled.")
            break

In [41]:
import time

In [42]:
print(time.time())

1755444733.7891884


In [43]:
if __name__ == "__main__":
    # Before running this script:
    # 1. Make sure you have Ollama installed and running.
    # 2. Pull the required models by running 'ollama pull llama3' and 'ollama pull nomic-embed-text'
    #    in your terminal.
    # 3. Place your PDF file in the same directory as this script and rename it to 'your_document.pdf'.
    
    # Start the chatbot
    print(time.time())
    chat_with_pdf()
    print(time.time())


1755444734.318271
A RAG vector store for 'The styrene production' already exists at './The styrene production_rag_db'.


Do you want to (a)dd to it, (r)emove and replace it, or (c)ontinue with old RAG? (a/r/c):  c


Continuing with the existing vector store...

-------------------------------------------
Chatbot is ready. You can now ask questions about 'The styrene production.pdf'.
Type 'exit' to quit.
-------------------------------------------



Your question:  explain the process


Thinking...

Chatbot: The styrene production process involves several steps:

1. Preheating: The feed mixture of benzene and ethylene is preheated to 95°C at a constant pressure of 1.01325 bar.
2. Pressurization: The preheated feed is sent to a pump to increase its pressure from 1.01325 bar to 13 bar.
3. Conversion reaction: The pressurized feed enters a conversion reactor, where ethyl benzene is produced with a 70% conversion rate based on benzene.
4. Cooling: The product stream from the reactor, containing ethyl benzene, unreacted ethylene, and benzene, is directed to a cooler that reduces its temperature to 150°C.
5. Separation: After cooling, the stream proceeds to a first distillation column, which separates purified ethyl benzene as the bottom product from the distillate containing unreacted ethylene and benzene.

This process converts benzene and ethylene into ethyl benzene, which is the primary component of styrene.



Your question:  exit


Goodbye!
1755445070.057687


In [ ]:
"""
prompts
now consider you have modeled pump, heater , compressor , reactor , seperator , distilaltion column, seperator and other few equipemnts 

1. **Pump**:
	* Feed from storage tank (at 1 atm) is pumped to raise pressure to 13 atm.
	* Stream leaving pump: S1
2. **Heater**:
	* S1 is heated up from 35°C to 190°C to meet desired conditions for ethylene glycol production.
	* Heated stream: S2
3. **Reactor**:
	* S2 (containing EO and water) enters the reactor, where reactions occur to produce EG and DEG.
	* After reactions are completed, liquid products and vapors are produced.
4. **Mixer**:
	* Vapors and liquid products from reactor are mixed to ensure equal composition.
5. **Control Valve**:
	* Mixed stream is passed through a control valve to reduce pressure before entering the distillation column (atmospheric distillation).
6. **Distillation Column**:
	* Distillation separates EG, DEG, and water from the mixture.
7. **Separator** (not explicitly mentioned in the context, but assumed as part of the distillation process):
	* Separates the products into different streams.


"""

In [ ]:
import operator
from typing import TypedDict, Sequence, Annotated

# Import necessary components from langchain and langgraph
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_community.chat_models import ChatOllama
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode

# 1. Define the AgentState
# The 'messages' key will accumulate BaseMessage objects.
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    # We'll add a 'context' field to demonstrate injecting arbitrary data
    context: Annotated[str | None, operator.setitem] # Use setitem to replace context

# 2. Define a simple tool
@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two integers together."""
    return a * b

tools = [multiply]

# 3. Initialize the Language Model
# Ensure Ollama is running and 'llama3.2' model is available
try:
    llm = ChatOllama(model='llama3.2', temperature=0.7).bind_tools(tools)
    model = llm
except Exception as e:
    print(f"Could not connect to Ollama or model not found: {e}")
    print("Falling back to a dummy model for demonstration purposes.")
    # Fallback for demonstration if Ollama isn't set up
    from langchain_core.runnables import RunnableLambda
    class DummyAIMessage(AIMessage):
        tool_calls = [] # Dummy for demonstration without actual tool calls

    def dummy_invoke(messages):
        # Simulate an LLM response based on user input
        last_user_message = next((m for m in reversed(messages) if isinstance(m, HumanMessage)), None)
        if last_user_message and "multiply" in last_user_message.content.lower():
            # Simulate a tool call if 'multiply' is mentioned
            return DummyAIMessage(
                content="Okay, let me calculate that for you.",
                tool_calls=[{"name": "multiply", "args": {"a": 5, "b": 3}}]
            )
        return DummyAIMessage(content=f"Hello, I received your message: '{last_user_message.content if last_user_message else 'None'}'. How can I help?")

    model = RunnableLambda(dummy_invoke).bind_tools(tools)


# 4. Define Graph Nodes

# Node for LLM interaction
def call_model(state: AgentState) -> AgentState:
    """Calls the LLM with the current messages and returns its response."""
    print(f"\n--- call_model node ---")
    print(f"Current messages before LLM call: {[m.content for m in state['messages']]}")
    print(f"Current context: {state.get('context', 'No context')}")

    # Add a system prompt for context
    system_prompt = SystemMessage(content="You are a helpful assistant. Use tools when appropriate.")
    messages_to_send = [system_prompt] + state['messages']

    response = model.invoke(messages_to_send)
    print(f"LLM response: {response.content}")
    return {"messages": [response]}

# Conditional edge logic
def should_continue(state: AgentState) -> str:
    """
    Determines whether to continue to tools or end the conversation.
    If the last message contains tool calls, continue; otherwise, end.
    """
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        print(f"--- should_continue: detected tool calls, continuing to 'tools' ---")
        return "continue"
    else:
        print(f"--- should_continue: no tool calls, ending ---")
        return "end"

# 5. Build the LangGraph
graph = StateGraph(AgentState)

graph.add_node("llm_node", call_model)
tool_node = ToolNode(tools=tools)
graph.add_node("tool_executor", tool_node)

# Define the workflow
graph.add_edge(START, "llm_node")
graph.add_conditional_edges(
    "llm_node",
    should_continue,
    {"continue": "tool_executor", "end": END}
)
graph.add_edge("tool_executor", "llm_node") # After tool execution, go back to LLM for summary

app = graph.compile()

# Helper function to print the stream
def print_stream(stream):
    print("\n--- Stream Output ---")
    for s in stream:
        print(s)
        print("-" * 20)
    print("--- End Stream Output ---\n")

# --- Demonstrating injectState ---

print("--- Scenario 1: Starting fresh (no injectState) ---")
initial_input = {"messages": [HumanMessage(content="What is 7 + 8?")]}
stream1 = app.stream(initial_input, stream_mode="values")
print_stream(stream1)


print("\n--- Scenario 2: Starting with pre-defined messages using injectState ---")
# Here, we're providing an initial state with existing messages.
# The graph will start as if these messages were already exchanged.
pre_existing_messages = [
    HumanMessage(content="Hello there!"),
    AIMessage(content="Hi! How can I assist you today?"),
    HumanMessage(content="I need to know what 12 multiplied by 5 is.")
]
# injectState is typically used with `.invoke()` or `.stream()` as an argument
# or can be used with `.get_graph().invoke/stream` if you have a compiled graph
# to set the initial state.
# For app.stream, you can pass the initial state directly in the inputs dict
# This is functionally similar to injectState when starting from START.
stream2_input = {
    "messages": pre_existing_messages
}
stream2 = app.stream(stream2_input, stream_mode="values")
print_stream(stream2)


print("\n--- Scenario 3: Simulating resuming a conversation (injecting state mid-way) ---")
# Imagine we saved the state after the LLM made a tool call and before the tool executed.
# Now we want to resume and let the tool execute.
# For simplicity, we'll manually construct a state that looks like the graph
# just finished `llm_node` and is about to enter `tool_executor`.
# In a real scenario, you'd save the actual state from a previous run.
saved_partial_state = {
    "messages": [
        HumanMessage(content="Please multiply 10 by 20."),
        AIMessage(content="Okay, I can do that.", tool_calls=[{"name": "multiply", "args": {"a": 10, "b": 20}}])
    ],
    "context": "Resumed from a checkpoint after LLM decided to use a tool."
}

# When resuming from a specific node, you use the 'from' argument.
# When `from` is specified, `injectState` essentially acts as the starting point,
# overriding the default START node and starting execution from the specified node
# with the provided state.
# Note: For `app.stream`, you'd typically pass the state as the *initial* input.
# The concept of `injectState` is more explicit when building dynamic graphs
# or resuming from exact states in the state machine using `get_graph().invoke(input, config={'recursion_limit': N}, state=saved_state)`.
# For the `app.stream` method of a compiled graph, injecting a specific 'initial'
# state means simply providing it in the input dictionary. LangGraph will ensure
# the state is correctly processed from the START node.

print("This demonstration directly sets the input as a 'resumed' state for stream.")
print("In a real application, you'd load a complete state object and use it with app.invoke/stream directly.")

stream3_input = saved_partial_state # This acts as the injected state for a new stream
stream3 = app.stream(stream3_input, stream_mode="values")
print_stream(stream3)

print("\n--- Scenario 4: Injecting only context ---")
# Start a new conversation, but provide some background context directly via state.
stream4_input = {
    "messages": [HumanMessage(content="What's the capital of France?")],
    "context": "This user is interested in geography."
}
stream4 = app.stream(stream4_input, stream_mode="values")
print_stream(stream4)



In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage],add_messages]

@tool
def add(a: int, b: int) -> int:
    """This is an addition function that adds two integers."""
    return a + b
@tool
def sub(a,b):
    "this is a subtraction function"
    return a-b

@tool
def mult(a,b):
    'this is a multiplacation function'
    return a*b

@tool
def heater_energy(mass_flow:float,heat_cap:float,in_temp:float,out_temp:float):
    """heat energy if calculated in this function where  mass_flow is the mass of the material flow , heat_cap is the heat capacity of the materia
    ,in_temp is the inlet temperature , out_temp is the outlet temperature of the stream
    """
    return mass_flow*heat_cap*(out_temp-in_temp)

tools = [add,sub,mult,heater_energy]


llm=init_chat_model(model='llama3.2',model_provider="ollama",  # Explicitly state the provider
    temperature=0.7,).bind_tools(tools)



def model_call(state: AgentState) -> AgentState:
    """
    Invokes the language model with the current messages and returns its response.
    """
    system_prompt = SystemMessage(content='You are my AI assistance, please answer my query.')
    # Concatenate the system prompt with the current messages
    response = llm.invoke([system_prompt] + state['messages'])
    return {'messages': [response]}

def should_continue(state: AgentState) -> str:
    """
    Determines whether to continue to tools or end the conversation.
    If the last message contains tool calls, continue to tools; otherwise, end.
    """
    message = state['messages']
    last_message = message[-1]
    if not last_message.tool_calls:
        return 'end'
    else:
        return 'continue'




graph = StateGraph(AgentState)
graph.add_node('our_agent', model_call) # Node for LLM interaction
tool_node = ToolNode(tools=tools) # Node for tool execution
graph.add_node('tools', tool_node) # Node for tool execution
graph.add_edge(START, 'our_agent') # Start the graph by calling our agent
graph.add_conditional_edges(
    'our_agent',
    should_continue,
    {
        'continue': 'tools', # If tool calls are present, go to 'tools' node
        'end': END           # If no tool calls, end the graph
    },
)

graph.add_edge('tools', 'our_agent') # After tool execution, go back to 'our_agent'

# Compile the graph to create the runnable application
app = graph.compile()

# def print_stream(stream):
#     for s in stream :
#         message=s['messages'][-1]
#         if isinstance(message,tuple):
#             print(message)

#         else:
#             message.pretty_print()

def print_stream(stream):
    """
    Iterates through the stream of states and prints the last message in each state.
    """
    for s in stream:
        # Corrected typo: 'messsages' -> 'messages'
        message = s['messages'][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

# inputs={'messages':[{'user','Add 3 + 4.'}]}
# # inputs = {'messages': [HumanMessage(content="Add 3+4")]}
# # print_stream(app.stream(inputs,stream_mode='values'))
# print_stream(app.stream(inputs, stream_mode='values'))

# Example input for the graph
# Corrected message format: use HumanMessage object
inputs = {'messages': [HumanMessage(content="Add 3+4 , subtract 5 from 3 and multiply 15 times 10 , 10kg of water is entering a heater and the heat capacity of water is 4.18 it enters at 35 C and exit at 80C what is the energy produced")]}

# Run the application and print the stream
print_stream(app.stream(inputs, stream_mode='values'))


In [45]:
# chemical_fraction={'eg':0.1,'b':0.25,'tg':0.5}
# chemicals_list=['eg','b','tg','vv']
# mole_frac={}
# for i in chemicals_list:
#     if i in chemical_fraction.keys():
#         mole_frac[i] = chemical_fraction[i]
#     else:
#         mole_frac[i] = 0

# mole_frac

{'eg': 0.1, 'b': 0.25, 'tg': 0.5, 'vv': 0}

In [46]:
def stream(name:str,temperature:float,pressure:float,chemicals_list:list,chemical_fraction:dict,flow:float):
    """Stream function is used to define the stream  
    name = name of the stream
    temperature = temperature of the stream
    pressure=pressure of the stream,
    chemical_list = list of all the chemicals used in the process
    chemical_fraction = flowrate of the input chemicals or molefraction of all the input chemicals 
    flow= flowrate of the stream mole flow 
    """
    mole_frac={}
    for i in chemicals_list:
        if i in chemical_fraction.keys():
            mole_frac[i] = chemical_fraction[0]
        else:
            mole_frac[i] = 0
    stream={
        'Name': name, 
 'Temperature (K)': temperature, 
 'Pressure (Pa)': pressure,
 'Molar Flowrate (mol/s)': flow,
    'Mole Fraction':mole_frac
    }
    return stream

def heater(inlet_stream:dict,heater_type:str,condition:float,name=str):
    """Heater is a heating unit where the temperature of the inlet stream is increased 
    inlet_stream = stream class is sent as 
    heater_type=heater type based on data whether 'outlet_temperature' or 'temperature_difference',
    condition=value of outlet temperature or temperature difference 
    name=Name of the outlet stream from the heater
    """
    # heat_capacity=inlet_stream.cp()
    # flow_rate=inlet_stream.flow()
    outlet_stream=inlet_stream.copy()
    if heater_type =='outlet_temperature':
        outlet_temp= condition
        temperature_diff= condition -inlet_stream['Temperature (K)'] 
    else:
        temperature_diff=condition
        outlet_temp= inlet_stream['Temperature (K)'] +condition
    energy =inlet_stream['Molar Flowrate (mol/s)']*4.31*temperature_diff
    outlet_stream['Temperature (K)']=outlet_temp
    outlet_stream['Name']=name
    return  ioutlet_stream

def pump(inlet_stream:dict,pump_condition:str,value:float,name:str):
    """Pump is a pumping unit where the pressure of the inlet stream is increased 
    inlet_stream = stream dict is sent 
    pump_condition=pump condition based on data whether 'outlet_pressure' or 'pressure_difference' is given in the data,
    value=value of outlet temperature or temperature difference 
    """
    outlet_stream=inlet_stream.copy()
    if pump_condition =='outlet_pressure':
        outlet_temp= value
        pressure_diff= value -inlet_stream['Pressure (Pa)'] 
    else:
        prespressuresure_diff=value
        outlet_temp= inlet_stream['Pressure (Pa)'] +value
    energy =inlet_stream['Molar Flowrate (mol/s)']*4.31*100
    outlet_stream['Pressure (Pa)']=outlet_temp
    outlet_stream['Name']=name
    return  outlet_stream

def Flash(inlet_stream:dict,pressure_drop:bool,value=0,gas_name=str,liq_name=str):
    """Flash is a stream seperator into liquid and vapor  
    inlet_stream = stream dict is sent 
    pressure_drop=to specify there is pressure changes in flash unit if true specify the pressure in flash,
    value=when the pressure drop is true specify the pressure in flash tank  
    """
    if pressure_drop ==True:
        outlet_temp= value
        pressure_diff= value -inlet_stream['Pressure (Pa)'] 

    inlet_stream['Pressure (Pa)']=outlet_temp
    inlet_stream1=inlet_stream.copy()
    inlet_stream1['Name']=gas_name
    inlet_stream2=inlet_stream.copy()
    inlet_stream2['Name']=liq_name

    return  inlet_stream1 ,inlet_stream2

def Reactor(inlet_stream:dict,Reaction_stoichiometry:dict,conversion=float,name=str):
    """Reactor is a  conversion reactpor 
    stream = reactor inlet stream (containing temperature , pressure , mole flow , mole fraction and other detail) 
    Reaction_stoichiometry = in a order structure chemicals involved in dictionary as key and the stoichimetry coefficient as value for reactant its negative and product its positive,
    conversion=conversion percentage of the reaction
    name= name of the outlet stream from reactor
    """
    outlet_stream=inlet_stream.copy()
    print('conversion',conversion)
    print('Reaction_stoichiometry')
    outlet_stream['conversion']=conversion
    outlet_stream['Reaction_stoichiometry']=Reaction_stoichiometry
    outlet_stream['Name']=name
    return  outlet_stream

In [ ]:
# def stream(name:str,temperature:float,pressure:float,chemicals_list:list,chemical_fraction:dict,flow:float):
#     """Stream function is used to define the stream  
#     name = name of the stream
#     temperature = temperature of the stream
#     pressure=pressure of the stream,
#     chemical_list = list of all the chemicals used in the process
#     chemical_fraction = flowrate of the input chemicals or molefraction of all the input chemicals 
#     flow= flowrate of the stream mole flow 
#     """
#     mole_frac={}
#     for i in chemicals_list:
#         if i in chemical_fraction.keys():
#             mole_frac[i] = chemical_fraction[0]
#         else:
#             mole_frac[i] = 0
        
#     a={'Name': name, 
#  'Temperature (K)': temperature, 
#  'Pressure (Pa)': pressure,
#  'Molar Flowrate (mol/s)': 10.0,
#  'Mixture Molar Weight (g/mol)': 18.0153,
#  'Phase': 'two-phase', 
#  'Vapor Fraction': 1.0,
#  'Mixture Molar Cp (J/mol.K)': 20786.0,
#  'Mixture Molar Enthalpy (J/mol)': -241812423381.9,
#  'Total Enthalpy Flow (W)': -2418124233819.0,
#  'Mixture Molar Heat of Vaporization (J/mol)': 0.0,
#  'Liquid Phase Mole Fractions': {'Water': 1.0, 'Methane': 0.0},
#  'Liquid Phase Molar Cp (J/mol.K)': 2365796.9325776, 
#  'Liquid Phase Molar Enthalpy (J/mol)': -241673416234.23148,
#  'Liquid Phase Density (kg/m^3)': None,
#  'Liquid Phase Thermal Conductivity (W/m.K)': -359521.62635176,
#  'Vapor Phase Mole Fractions': {'Water': 1.0, 'Methane': 0.0},
#  'Vapor Phase Molar Cp (J/mol.K)': 20786.0,
#  'Vapor Phase Molar Enthalpy (J/mol)': -241812423381.9,
#  'Vapor Phase Thermal Conductivity (W/m.K)': 732007429907.3121}

In [21]:
def heater(inlet_stream:dict,heater_type:str,condition:float):
    """Heater is a heating unit where the temperature of the inlet stream is increased 
    inlet_stream = stream class is sent as 
    heater_type=heater type based on data whether 'outlet_temperature' or 'temperature_difference',
    condition=value of outlet temperature or temperature difference 
    """
    # heat_capacity=inlet_stream.cp()
    # flow_rate=inlet_stream.flow()
    if heater_type =='outlet_temperature':
        outlet_temp= condition
        temperature_diff= condition -inlet_stream['Temperature (K)'] 
    else:
        temperature_diff=condition
        outlet_temp= inlet_stream['Temperature (K)'] +condition
    energy =inlet_stream['Molar Flowrate (mol/s)']*4.31*temperature_diff
    inlet_stream['Temperature (K)']=outlet_temp
    return  inlet_stream

heater(a,'outlet_temperature',500)

{'Name': 'Two-Phase Stream',
 'Temperature (K)': 500,
 'Pressure (Pa)': 201325.0,
 'Molar Flowrate (mol/s)': 10.0,
 'Mixture Molar Weight (g/mol)': 18.0153,
 'Phase': 'two-phase',
 'Vapor Fraction': 1.0,
 'Mixture Molar Cp (J/mol.K)': 20786.0,
 'Mixture Molar Enthalpy (J/mol)': -241812423381.9,
 'Total Enthalpy Flow (W)': -2418124233819.0,
 'Mixture Molar Heat of Vaporization (J/mol)': 0.0,
 'Liquid Phase Mole Fractions': {'Water': 1.0, 'Methane': 0.0},
 'Liquid Phase Molar Cp (J/mol.K)': 2365796.9325776,
 'Liquid Phase Molar Enthalpy (J/mol)': -241673416234.23148,
 'Liquid Phase Density (kg/m^3)': None,
 'Liquid Phase Thermal Conductivity (W/m.K)': -359521.62635176,
 'Vapor Phase Mole Fractions': {'Water': 1.0, 'Methane': 0.0},
 'Vapor Phase Molar Cp (J/mol.K)': 20786.0,
 'Vapor Phase Molar Enthalpy (J/mol)': -241812423381.9,
 'Vapor Phase Thermal Conductivity (W/m.K)': 732007429907.3121}

In [50]:
# def pump(inlet_stream:dict,pump_condition:str,value:float):
#     """Pump is a pumping unit where the pressure of the inlet stream is increased 
#     inlet_stream = stream dict is sent 
#     pump_condition=pump condition based on data whether 'outlet_pressure' or 'pressure_difference' is given in the data,
#     value=value of outlet temperature or temperature difference 
#     """
#     # heat_capacity=inlet_stream.cp()
#     # flow_rate=inlet_stream.flow()
#     if pump_condition =='outlet_pressure':
#         outlet_temp= value
#         pressure_diff= value -inlet_stream['Pressure (Pa)'] 
#     else:
#         prespressuresure_diff=value
#         outlet_temp= inlet_stream['Pressure (Pa)'] +value
#     energy =inlet_stream['Molar Flowrate (mol/s)']*4.31*100
#     inlet_stream['Pressure (Pa)']=outlet_temp
#     return  inlet_stream

# pump(a,'outlet_pressure',201325.0)

In [30]:
def Flash(inlet_stream:dict,pressure_drop:bool,value=0):
    """Flash is a stream seperator into liquid and vapor  
    inlet_stream = stream dict is sent 
    pressure_drop=to specify there is pressure changes in flash unit if true specify the pressure in flash,
    value=when the pressure drop is true specify the pressure in flash tank  
    """
    # heat_capacity=inlet_stream.cp()
    # flow_rate=inlet_stream.flow()
    if pressure_drop ==True:
        outlet_temp= value
        pressure_diff= value -inlet_stream['Pressure (Pa)'] 

    # energy =inlet_stream['Molar Flowrate (mol/s)']*4.31*temperature_diff
    inlet_stream['Pressure (Pa)']=outlet_temp
    inlet_stream['Name']='gas'
    inlet_stream1=inlet_stream.copy()
    inlet_stream['Name']='liquid'
    inlet_stream2=inlet_stream.copy()
    # inlet_stream2=inlet_stream
    return  inlet_stream1 ,inlet_stream2

Flash(a,True,201325.0)

({'Name': 'gas',
  'Temperature (K)': 500,
  'Pressure (Pa)': 201325.0,
  'Molar Flowrate (mol/s)': 10.0,
  'Mixture Molar Weight (g/mol)': 18.0153,
  'Phase': 'two-phase',
  'Vapor Fraction': 1.0,
  'Mixture Molar Cp (J/mol.K)': 20786.0,
  'Mixture Molar Enthalpy (J/mol)': -241812423381.9,
  'Total Enthalpy Flow (W)': -2418124233819.0,
  'Mixture Molar Heat of Vaporization (J/mol)': 0.0,
  'Liquid Phase Mole Fractions': {'Water': 1.0, 'Methane': 0.0},
  'Liquid Phase Molar Cp (J/mol.K)': 2365796.9325776,
  'Liquid Phase Molar Enthalpy (J/mol)': -241673416234.23148,
  'Liquid Phase Density (kg/m^3)': None,
  'Liquid Phase Thermal Conductivity (W/m.K)': -359521.62635176,
  'Vapor Phase Mole Fractions': {'Water': 1.0, 'Methane': 0.0},
  'Vapor Phase Molar Cp (J/mol.K)': 20786.0,
  'Vapor Phase Molar Enthalpy (J/mol)': -241812423381.9,
  'Vapor Phase Thermal Conductivity (W/m.K)': 732007429907.3121},
 {'Name': 'liquid',
  'Temperature (K)': 500,
  'Pressure (Pa)': 201325.0,
  'Molar Flowr

In [ ]:
class stream_spec():
    def __init__(self,name,Temperature:float,pressure:float,composition:list,flow_rate:list)
        self.name=name
        self.Temperature=Temperature
        self.pressure=pressure
        self.composition=composition
        self.flow_rate=flow_rate

def heater(name:str,inlet_stream:class,heater_type:str,condition:float):
    """Heater is a heating unit where the temperature of the inlet stream is increased 
    name = name of the heater unit 
    inlet_stream = stream class is sent as 
    heater_type=heater type based on data whether 'outlet_temperature' or 'temperature_difference',
    condition=value of outlet temperature or temperature difference 
    """
    flow_rate=inlet_stream.flow_rate()
    if heater_type =='outlet_temperature':
        temperature_out=inlet_stream.Temperature() - condition
    else:
        temperature_out=condition + condition
    heater=stream_spec(f'{name}_outlet',temperature_out,inlet_stream.pressure(),inlet_stream.composition(),inlet_stream.flow_rate())
     return heater

def pump(name:str,inlet_stream:class,heater_type:str,condition:float):
    """Heater is a heating unit where the temperature of the inlet stream is increased 
    name = name of the heater unit 
    inlet_stream = stream class is sent as 
    heater_type=heater type based on data whether 'outlet_temperature' or 'temperature_difference',
    condition=value of outlet temperature or temperature difference 
    """
    flow_rate=inlet_stream.flow_rate()
    if heater_type =='outlet_temperature':
        temperature_out=inlet_stream.Temperature() - condition
    else:
        temperature_out=condition + condition
    heater=stream_spec(f'{name}_outlet',temperature_out,inlet_stream.pressure(),inlet_stream.composition(),inlet_stream.flow_rate())
     return heater